# FedVocabHAR WISDM Phase Zero-D: Multimodal Identity Gate

Phase Zero-C showed that conditional calibration cannot repair insufficient sensor discrimination. The dominant errors—O/S, O/P, P/S, M/S, and M/O—are similar hand-motion activities. This notebook therefore evaluates the scientifically justified sensor upgrade: synchronized smartwatch **accelerometer + gyroscope** signals.

The protocol remains locked: development subjects and activities train the representation and calibrate the matcher; engineering subjects and novel activities are used only for scoring. Signal-statistics, self-supervised multimodal prototypes, and their hybrid are compared.

**Pair gate:** AUROC ≥ 0.85, TPR ≥ 0.70, FPR ≤ 0.05.  
**Registry gate:** precision ≥ 0.90 and recall ≥ 0.70 under independent false-novel reports.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from dataclasses import dataclass, asdict
from pathlib import Path
import copy, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

@dataclass(frozen=True)
class Config:
    project: str = '/content/drive/MyDrive/FedVocabHAR'
    seed: int = 42
    known: tuple = ('A','B','C','D','E','R')
    development_novel: tuple = ('F','G','H','I','K')
    engineering_novel: tuple = ('L','M','O','P','Q','S')
    development_subjects: tuple = tuple(range(1635,1642))
    engineering_subjects: tuple = tuple(range(1643,1651))
    sampling_rate: float = 20.0
    ssl_epochs: int = 30
    batch_size: int = 256
    embedding_dim: int = 96
    temperature: float = 0.15
    target_fpr: float = 0.05
    registry_margin: float = 0.05
    registry_trials: int = 100

CFG=Config(); PROJECT=Path(CFG.project)
OUT=PROJECT/'artifacts'/'phase0d'; OUT.mkdir(parents=True,exist_ok=True)
ACCEL=PROJECT/'data'/'WISDM-2019'/'raw'/'watch'/'accel'
GYRO=PROJECT/'data'/'WISDM-2019'/'raw'/'watch'/'gyro'
CACHE=OUT/'wisdm_watch_accel_gyro_w128_s64.npz'
assert ACCEL.exists() and GYRO.exists(), 'Missing WISDM smartwatch accel/gyro folders.'

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all(CFG.seed)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:',DEVICE,'project:',PROJECT)

device: cuda project: /content/drive/MyDrive/FedVocabHAR


## Build or load synchronized multimodal windows

Accelerometer and gyroscope samples are joined by their recorded timestamps within each subject and activity. Windows never cross an activity boundary. The six-channel cache is retained in Drive for reproducibility.

In [3]:
from collections import defaultdict

def parse_sensor(path):
    groups=defaultdict(list)
    with path.open('r',encoding='utf-8',errors='ignore') as f:
        for line in f:
            q=line.strip().rstrip(';').split(',')
            if len(q)!=6: continue
            try: groups[q[1]].append((int(q[2]),float(q[3]),float(q[4]),float(q[5])))
            except ValueError: pass
    return {a:np.asarray(v,dtype=np.float64) for a,v in groups.items()}

def build_multimodal_cache():
    xs=[]; ss=[]; aa=[]; pp=[]
    wanted=set(CFG.known+CFG.development_novel+CFG.engineering_novel)
    for s in CFG.development_subjects+CFG.engineering_subjects:
        acc=parse_sensor(ACCEL/f'data_{s}_accel_watch.txt'); gyr=parse_sensor(GYRO/f'data_{s}_gyro_watch.txt')
        for a in sorted(wanted & set(acc) & set(gyr)):
            av,gv=acc[a],gyr[a]
            common,ia,ig=np.intersect1d(av[:,0].astype(np.int64),gv[:,0].astype(np.int64),return_indices=True)
            values=np.concatenate([av[ia,1:4],gv[ig,1:4]],axis=1).astype(np.float32)
            for start in range(0,len(values)-128+1,64):
                xs.append(values[start:start+128].T); ss.append(s); aa.append(a); pp.append(start/max(1,len(values)-128))
        print(s,'windows so far',len(xs))
    np.savez_compressed(CACHE,x=np.asarray(xs,dtype=np.float32),subject=np.asarray(ss),activity=np.asarray(aa),position=np.asarray(pp,dtype=np.float32))

if not CACHE.exists(): build_multimodal_cache()
d=np.load(CACHE); x=d['x'].astype(np.float32); subjects=d['subject']; activities=d['activity']; positions=d['position']
dev_mask=np.isin(subjects,CFG.development_subjects); eng_mask=np.isin(subjects,CFG.engineering_subjects)
assert x.shape[1]==6, x.shape
print('multimodal windows:',x.shape,'development:',dev_mask.sum(),'engineering:',eng_mask.sum())
print(pd.crosstab(subjects,activities).to_string())

1635 windows so far 935
1636 windows so far 1870
1637 windows so far 2081
1638 windows so far 2408
1639 windows so far 2696
1640 windows so far 3087
1641 windows so far 4022
1643 windows so far 4957
1644 windows so far 5977
1645 windows so far 6957
1646 windows so far 7892
1647 windows so far 8837
1648 windows so far 9772
1649 windows so far 10712
1650 windows so far 11674
multimodal windows: (11674, 6, 128) development: 4022 engineering: 7652
col_0   A   B   C   D   E   F   G   H   I   K   L   M   O   P   Q   R   S
row_0                                                                    
1635   55  55  55  55  55  55  55  55  55  55  55  55  55  55  55  55  55
1636   55  55  55  55  55  55  55  55  55  55  55  55  55  55  55  55  55
1637    1   0   0  29  33   7  12  19  50  22  17   0   0   0   2  19   0
1638   33  34  35  60  14   0   0   0   0  15   2  37   0  35   0  30  32
1639    0   3  33   0  64   0  15  37   1  37   0   0   0   2  51  33  12
1640   28  28   0  33   0   0  60 

## Representation A: multimodal signal statistics

Time- and frequency-domain features are computed separately for accelerometer axes, gyroscope axes, and both sensor magnitudes. Within-sensor axis correlations and the acceleration–rotation magnitude correlation are also included.

In [4]:
def signal_features(x,fs=20.0):
    amag=np.linalg.norm(x[:,:3],axis=1,keepdims=True); gmag=np.linalg.norm(x[:,3:6],axis=1,keepdims=True)
    z=np.concatenate([x,amag,gmag],axis=1); centered=z-z.mean(2,keepdims=True); std=centered.std(2)+1e-8; feats=[]
    funcs=[lambda a:a.mean(2),lambda a:a.std(2),lambda a:np.median(a,2),lambda a:np.quantile(a,.10,axis=2),
           lambda a:np.quantile(a,.25,axis=2),lambda a:np.quantile(a,.75,axis=2),lambda a:np.quantile(a,.90,axis=2),
           lambda a:np.sqrt((a*a).mean(2)),lambda a:np.abs(a).mean(2),lambda a:np.sqrt((np.diff(a,axis=2)**2).mean(2))]
    feats += [fn(z) for fn in funcs]
    feats.append((np.diff(np.signbit(centered),axis=2)!=0).mean(2))
    feats.append((centered[:,:,:-1]*centered[:,:,1:]).mean(2)/(std*std))
    fft=np.abs(np.fft.rfft(centered,axis=2))**2; freqs=np.fft.rfftfreq(z.shape[2],1/fs); fft[:,:,0]=0
    total=fft.sum(2)+1e-8; p=fft/total[:,:,None]
    feats += [(fft*freqs).sum(2)/total,freqs[np.argmax(fft,axis=2)],-(p*np.log(p+1e-12)).sum(2)]
    for lo,hi in [(0,.5),(.5,1.5),(1.5,3),(3,5),(5,10.1)]:
        band=(freqs>=lo)&(freqs<hi); feats.append(fft[:,:,band].sum(2)/total)
    corr=[]
    for base in (0,3):
        corr += [(centered[:,base+i]*centered[:,base+j]).mean(1)/(std[:,base+i]*std[:,base+j]) for i,j in ((0,1),(0,2),(1,2))]
    corr.append((centered[:,6]*centered[:,7]).mean(1)/(std[:,6]*std[:,7]))
    feats.append(np.stack(corr,1))
    return np.concatenate(feats,1).astype(np.float32)

sf=signal_features(x,CFG.sampling_rate); print('multimodal signal feature matrix:',sf.shape)

multimodal signal feature matrix: (11674, 167)


## Representation B: multimodal self-supervised encoder

The contrastive encoder is retrained with six synchronized channels. It sees development-client windows without activity labels and excludes all engineering clients.

In [5]:
channel_mean=x[dev_mask].mean((0,2),keepdims=True); channel_std=x[dev_mask].std((0,2),keepdims=True)+1e-6
xn=(x-channel_mean)/channel_std

def augment(a):
    y=a.clone(); y=y*(.8+.4*torch.rand(y.size(0),y.size(1),1,device=y.device))
    y=y+.04*torch.randn_like(y)
    if y.size(0)>1:
        shift=torch.randint(-8,9,(y.size(0),),device=y.device)
        y=torch.stack([torch.roll(v,int(s.item()),dims=-1) for v,s in zip(y,shift)])
    mask=torch.rand(y.size(0),1,y.size(2),device=y.device)<.06; y=y.masked_fill(mask,0)
    return y

class WindowDataset(Dataset):
    def __init__(self,array): self.array=torch.from_numpy(array)
    def __len__(self): return len(self.array)
    def __getitem__(self,i): return self.array[i]

class Encoder(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.body=nn.Sequential(nn.Conv1d(6,64,7,padding=3),nn.BatchNorm1d(64),nn.ReLU(),
          nn.Conv1d(64,96,5,stride=2,padding=2),nn.BatchNorm1d(96),nn.ReLU(),
          nn.Conv1d(96,128,5,stride=2,padding=2),nn.BatchNorm1d(128),nn.ReLU(),
          nn.Conv1d(128,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveAvgPool1d(1))
        self.project=nn.Sequential(nn.Linear(128,128),nn.ReLU(),nn.Linear(128,dim))
    def forward(self,z):
        h=self.body(z).squeeze(-1); return F.normalize(self.project(h),dim=1)

def ntxent(z1,z2,temp):
    n=len(z1); z=torch.cat([z1,z2],0); sim=z@z.T/temp
    sim.fill_diagonal_(-1e9); targets=torch.arange(2*n,device=z.device); targets=(targets+n)%(2*n)
    return F.cross_entropy(sim,targets)

ssl_pool=xn[dev_mask & np.isin(activities,CFG.known+CFG.development_novel)]
loader=DataLoader(WindowDataset(ssl_pool),batch_size=CFG.batch_size,shuffle=True,drop_last=True,num_workers=2)
encoder=Encoder(CFG.embedding_dim).to(DEVICE); opt=torch.optim.AdamW(encoder.parameters(),lr=8e-4,weight_decay=1e-4)
history=[]; best=None; best_loss=float('inf')
for epoch in range(CFG.ssl_epochs):
    encoder.train(); losses=[]
    for batch in loader:
        batch=batch.to(DEVICE); opt.zero_grad(); loss=ntxent(encoder(augment(batch)),encoder(augment(batch)),CFG.temperature)
        loss.backward(); opt.step(); losses.append(loss.item())
    mean_loss=float(np.mean(losses)); history.append({'epoch':epoch+1,'loss':mean_loss})
    if mean_loss<best_loss: best_loss=mean_loss; best=copy.deepcopy(encoder.state_dict())
    print(epoch+1,mean_loss)
encoder.load_state_dict(best); torch.save({'state_dict':best,'config':asdict(CFG),'channel_mean':channel_mean,'channel_std':channel_std},OUT/'contrastive_encoder.pt')

1 3.343599820137024
2 2.4697530269622803
3 2.1973132848739625
4 2.017434465885162
5 1.9058947324752809
6 1.8234485387802124
7 1.7675580859184266
8 1.719322681427002
9 1.6725557446479797
10 1.6377564787864685
11 1.6177126884460449
12 1.593120801448822
13 1.5912338972091675
14 1.5583460927009583
15 1.549630630016327
16 1.5165005326271057
17 1.5139376521110535
18 1.485812747478485
19 1.4518885135650634
20 1.4561142683029176
21 1.4511138081550599
22 1.4294015884399414
23 1.4420122146606444
24 1.413592028617859
25 1.401417088508606
26 1.4125959753990174
27 1.3867793202400207
28 1.3972970008850099
29 1.3520937085151672
30 1.3730119109153747


In [6]:
encoder.eval(); embeddings=[]
with torch.no_grad():
    for start in range(0,len(xn),1024): embeddings.append(encoder(torch.from_numpy(xn[start:start+1024]).to(DEVICE)).cpu().numpy())
emb=np.concatenate(embeddings); print('embedding matrix:',emb.shape)

embedding matrix: (11674, 96)


## Candidate descriptors and invariant pair features

Local clusters remain oracle-defined in Phase Zero-B. This isolates identity matching from novelty-detection and clustering errors. No coordinate-wise difference vector is given to the matcher; every feature is an invariant distance, similarity, or uncertainty ratio.

In [7]:
def aggregate(values,kind):
    if kind=='signal': return np.concatenate([np.median(values,0),values.mean(0),values.std(0)])
    mean=values.mean(0); mean/=np.linalg.norm(mean)+1e-12
    return np.concatenate([mean,values.std(0)])

def build_candidates(subject_list,activity_list):
    out={}
    for s in subject_list:
        for a in activity_list:
            m=(subjects==s)&(activities==a)
            if m.sum()>=3:
                out[(int(s),str(a))]={'signal':aggregate(sf[m],'signal'),'ssl':aggregate(emb[m],'ssl'),'support':int(m.sum())}
    return out

dev=build_candidates(CFG.development_subjects,CFG.development_novel)
eng=build_candidates(CFG.engineering_subjects,CFG.engineering_novel)
eng_known=build_candidates(CFG.engineering_subjects,CFG.known)

# Coordinate scaling is fitted only for distance conditioning, never exposed directly to the matcher.
for view in ('signal','ssl'):
    vals=np.stack([v[view] for v in dev.values()]); mu=vals.mean(0); sd=vals.std(0)+1e-8
    for pool in (dev,eng,eng_known):
        for item in pool.values(): item[view]=(item[view]-mu)/sd

def block_metrics(a,b,blocks):
    out=[]; start=0
    for width in blocks:
        x=a[start:start+width]; y=b[start:start+width]; d=x-y; start+=width
        out += [np.mean(np.abs(d)),np.sqrt(np.mean(d*d)),np.max(np.abs(d)),
                np.dot(x,y)/(np.linalg.norm(x)*np.linalg.norm(y)+1e-12),
                np.corrcoef(x,y)[0,1] if np.std(x)>0 and np.std(y)>0 else 0.0]
    return np.nan_to_num(np.asarray(out,dtype=np.float32))

signal_width=len(next(iter(dev.values()))['signal'])//3
ssl_width=len(next(iter(dev.values()))['ssl'])//2
def pair_features(a,b,variant):
    s=block_metrics(a['signal'],b['signal'],[signal_width]*3)
    z=block_metrics(a['ssl'],b['ssl'],[ssl_width]*2)
    extra=np.asarray([abs(math.log1p(a['support'])-math.log1p(b['support'])),math.log1p(min(a['support'],b['support']))],dtype=np.float32)
    return {'signal':np.r_[s,extra],'ssl':np.r_[z,extra],'hybrid':np.r_[s,z,extra]}[variant]

def pairs(pool,subjs,acts):
    rows=[]
    for i,s1 in enumerate(subjs):
        for s2 in subjs[i+1:]:
            for a in acts:
                for b in acts:
                    if (s1,a) in pool and (s2,b) in pool:
                        rows.append((pool[(s1,a)],pool[(s2,b)],int(a==b),int(s1),int(s2),str(a),str(b)))
    return rows

dev_pairs=pairs(dev,list(CFG.development_subjects),list(CFG.development_novel))
eng_pairs=pairs(eng,list(CFG.engineering_subjects),list(CFG.engineering_novel))
known_cal_pairs=pairs(eng_known,list(CFG.engineering_subjects),list(CFG.known))
print('candidates:',len(dev),len(eng),'pairs:',len(dev_pairs),len(eng_pairs))

candidates: 28 48 pairs: 329 1008


## Activity-held-out calibration and multimodal engineering evaluation

Complete development activities are excluded in turn for calibration. The final matcher trains on all development activities. Its threshold is the more conservative of activity-held-out calibration and mismatched known-activity calibration on engineering clients. No engineering novel label affects training or threshold selection.

In [8]:
def ece(y,p,bins=10):
    total=0.0
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        m=(p>=lo)&(p<(hi if hi<1 else 1.000001))
        if m.any(): total += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(total)

def make_model():
    return Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(max_iter=4000,class_weight='balanced',C=.1))])

models={}; thresholds={}; results=[]
for variant in ('signal','ssl','hybrid'):
    oof_y=[]; oof_p=[]
    for held in CFG.development_novel:
        train=[r for r in dev_pairs if r[5]!=held and r[6]!=held]
        valid=[r for r in dev_pairs if r[5]==held or r[6]==held]
        Xtr=np.stack([pair_features(a,b,variant) for a,b,*_ in train]); ytr=np.array([r[2] for r in train])
        Xv=np.stack([pair_features(a,b,variant) for a,b,*_ in valid]); yv=np.array([r[2] for r in valid])
        fold=make_model().fit(Xtr,ytr); oof_y.append(yv); oof_p.append(fold.predict_proba(Xv)[:,1])
    oof_y=np.concatenate(oof_y); oof_p=np.concatenate(oof_p)
    Xtr=np.stack([pair_features(a,b,variant) for a,b,*_ in dev_pairs]); ytr=np.array([r[2] for r in dev_pairs])
    Xte=np.stack([pair_features(a,b,variant) for a,b,*_ in eng_pairs]); yte=np.array([r[2] for r in eng_pairs])
    Xkc=np.stack([pair_features(a,b,variant) for a,b,*_ in known_cal_pairs]); ykc=np.array([r[2] for r in known_cal_pairs])
    model=make_model().fit(Xtr,ytr); pte=model.predict_proba(Xte)[:,1]; pkc=model.predict_proba(Xkc)[:,1]
    t_dev=float(np.quantile(oof_p[oof_y==0],1-CFG.target_fpr,method='higher'))
    t_known=float(np.quantile(pkc[ykc==0],1-CFG.target_fpr,method='higher'))
    threshold=max(t_dev,t_known); models[variant]=model; thresholds[variant]=threshold
    results.append({'variant':variant,'auroc':roc_auc_score(yte,pte),'average_precision':average_precision_score(yte,pte),
      'brier':brier_score_loss(yte,pte),'ece':ece(yte,pte),'threshold_dev':t_dev,'threshold_known':t_known,'threshold':threshold,
      'tpr':float((pte[yte==1]>=threshold).mean()),'fpr':float((pte[yte==0]>=threshold).mean()),
      'known_negative_fpr':float((pkc[ykc==0]>=threshold).mean())})
pair_results=pd.DataFrame(results); display(pair_results)

,variant,auroc,average_precision,brier,ece,threshold_dev,threshold_known,threshold,tpr,fpr,known_negative_fpr
0,signal,0.812457,0.398201,0.153279,0.153107,0.700982,0.739680,0.739680,0.375000,0.091667,0.050000
1,ssl,0.800702,0.618668,0.263127,0.390177,0.632968,0.708757,0.708757,0.666667,0.167857,0.050000
2,hybrid,0.840887,0.444191,0.143523,0.142300,0.722044,0.705072,0.722044,0.511905,0.092857,0.045238


## Conditional provisional-registry audit

The registry is evaluated only when the primary hybrid representation passes the pair gate. Each client contributes at most one report to an entry. Complete-link compatibility, a destination margin, and a three-client quorum are enforced. False-novel reports are drawn from known activities; mixture reports combine two different novel activities.

In [9]:
def compatibility(a,b,variant='hybrid'):
    return float(models[variant].predict_proba(pair_features(a,b,variant)[None])[0,1])

def mixed_signature(a,b):
    out={'support':min(a['support'],b['support'])}
    for view in ('signal','ssl'):
        out[view]=.5*a[view]+.5*b[view]
    return out

def registry_trial(mode,seed,quorum=3,variant='hybrid'):
    rng=np.random.default_rng(seed); reports=[]
    for activity in CFG.engineering_novel:
        for s in CFG.engineering_subjects:
            reports.append({'client':s,'truth':activity,'sig':eng[(s,activity)]})
    for s in CFG.engineering_subjects:
        if mode=='independent':
            a=rng.choice(CFG.known); sig=eng_known[(s,a)]; truth=f'false-known-{a}'
        else:
            a,b=rng.choice(CFG.engineering_novel,2,replace=False); sig=mixed_signature(eng[(s,a)],eng[(s,b)]); truth=f'mixture-{a}-{b}'
        reports.append({'client':s,'truth':truth,'sig':sig})
    rng.shuffle(reports); entries=[]; ambiguous=0
    for report in reports:
        choices=[]
        for idx,entry in enumerate(entries):
            if report['client'] in {r['client'] for r in entry}: continue
            scores=[compatibility(report['sig'],r['sig'],variant) for r in entry]
            if min(scores)>=thresholds[variant]: choices.append((float(np.mean(scores)),idx))
        choices.sort(reverse=True)
        if choices and (len(choices)==1 or choices[0][0]-choices[1][0]>=CFG.registry_margin): entries[choices[0][1]].append(report)
        else:
            if len(choices)>1: ambiguous+=1
            entries.append([report])
    promoted=[e for e in entries if len({r['client'] for r in e})>=quorum]
    correct=[]; recovered=set(); false_merge=0
    for e in promoted:
        truth={r['truth'] for r in e}; real={t for t in truth if len(t)==1}
        if len(real)==1 and len(truth)==1: correct.append(e); recovered|=real
        else: false_merge+=1
    return {'precision':len(correct)/len(promoted) if promoted else 0.0,'recall':len(recovered)/len(CFG.engineering_novel),
            'false_merge_rate':false_merge/len(promoted) if promoted else 0.0,'promoted':len(promoted),'ambiguous':ambiguous}

primary=pair_results.query("variant=='hybrid'").iloc[0]
pair_gate=bool(primary.auroc>=.85 and primary.tpr>=.70 and primary.fpr<=.05)
registry_results=pd.DataFrame()
if pair_gate:
    rows=[]
    for mode in ('independent','mixture'):
        trials=[registry_trial(mode,CFG.seed+i) for i in range(CFG.registry_trials)]
        rows.append({'mode':mode,**{k:float(np.mean([t[k] for t in trials])) for k in trials[0]}})
    registry_results=pd.DataFrame(rows); display(registry_results)
else:
    print('Registry audit skipped because the hybrid pair gate did not pass.')

Registry audit skipped because the hybrid pair gate did not pass.


## Save the decision record

In [10]:
registry_gate=False
if not registry_results.empty:
    rr=registry_results.query("mode=='independent'").iloc[0]
    registry_gate=bool(rr.precision>=.90 and rr.recall>=.70)
decision={'pair_gate':pair_gate,'registry_gate':registry_gate,'proceed_to_full_implementation':bool(pair_gate and registry_gate)}
manifest={'config':asdict(CFG),'device':str(DEVICE),'decision':decision,'note':'Multimodal WISDM engineering gate; reserve an external dataset for confirmatory evaluation.',
          'timestamp_utc':pd.Timestamp.utcnow().isoformat()}
pair_results.to_csv(OUT/'pair_results.csv',index=False)
registry_results.to_csv(OUT/'registry_results.csv',index=False)
pd.DataFrame(history).to_csv(OUT/'ssl_history.csv',index=False)
(OUT/'phase0b_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2)); print('saved to',OUT)

{
  "config": {
    "project": "/content/drive/MyDrive/FedVocabHAR",
    "seed": 42,
    "known": [
      "A",
      "B",
      "C",
      "D",
      "E",
      "R"
    ],
    "development_novel": [
      "F",
      "G",
      "H",
      "I",
      "K"
    ],
    "engineering_novel": [
      "L",
      "M",
      "O",
      "P",
      "Q",
      "S"
    ],
    "development_subjects": [
      1635,
      1636,
      1637,
      1638,
      1639,
      1640,
      1641
    ],
    "engineering_subjects": [
      1643,
      1644,
      1645,
      1646,
      1647,
      1648,
      1649,
      1650
    ],
    "sampling_rate": 20.0,
    "ssl_epochs": 30,
    "batch_size": 256,
    "embedding_dim": 96,
    "temperature": 0.15,
    "target_fpr": 0.05,
    "registry_margin": 0.05,
    "registry_trials": 100
  },
  "device": "cuda",
  "decision": {
    "pair_gate": false,
    "registry_gate": false,
    "proceed_to_full_implementation": false
  },
  "note": "Multimodal WISDM engineering gate;

## Interpretation

- **Pass both gates:** freeze the multimodal identity mechanism and proceed to novelty detection, temporal persistence, and risk-controlled integration.
- **AUROC passes but threshold metrics fail:** export pair scores and report a risk-coverage curve; do not claim reliable autonomous vocabulary birth yet.
- **Multimodal AUROC fails:** WISDM does not support sensor-only zero-label semantic identity at the required reliability. Revise the method toward human/semantic confirmation or broader pretrained representations before full implementation.